# Tutorial 13 — Classical ML Baseline for QSAR
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

Before reaching for deep learning, always build a strong classical ML baseline. Random Forest and XGBoost with Morgan fingerprints + physicochemical descriptors are surprisingly hard to beat on small molecular datasets (< 10k compounds).

In [ ]:
!pip install rdkit scikit-learn xgboost pandas numpy matplotlib -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

def featurize(smiles: str) -> np.ndarray | None:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    fp = np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048))
    pc = np.array([
        Descriptors.ExactMolWt(mol), Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol), rdMolDescriptors.CalcNumHBD(mol),
        rdMolDescriptors.CalcNumHBA(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        Descriptors.MolMR(mol), Descriptors.FractionCSP3(mol),
        Descriptors.NumValenceElectrons(mol),
    ])
    return np.concatenate([fp, pc])

# Simulated GPCR activity dataset
np.random.seed(42)
n_active, n_inactive = 100, 200
active_smiles   = ["CC(=O)Oc1ccccc1C(=O)O"] * n_active
inactive_smiles = ["Cn1cnc2c1c(=O)n(C)c(=O)n2C"] * n_inactive

all_smiles = active_smiles + inactive_smiles
labels = np.array([1]*n_active + [0]*n_inactive)
X = np.array([featurize(s) for s in all_smiles])
print(f"Feature matrix: {X.shape[0]} compounds × {X.shape[1]} features")
print(f"Class balance: {labels.sum()} active, {(labels==0).sum()} inactive")

In [ ]:
scaler = StandardScaler()
X_s = scaler.fit_transform(X)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Random Forest":    RandomForestClassifier(200, random_state=42, n_jobs=-1),
    "XGBoost":         XGBClassifier(200, random_state=42, eval_metric="auc", verbosity=0),
}

print("5-fold cross-validation results:")
print(f"{'Model':20s} {'AUC':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}")
print("-" * 50)
all_results = {}
for name, model in models.items():
    cv_res = cross_validate(model, X_s, labels, cv=cv,
                            scoring=["roc_auc","f1","precision","recall"])
    all_results[name] = cv_res
    print(f"{name:20s} {cv_res['test_roc_auc'].mean():.3f}±{cv_res['test_roc_auc'].std():.3f}"
          f" {cv_res['test_f1'].mean():.3f}±{cv_res['test_f1'].std():.3f}"
          f" {cv_res['test_precision'].mean():.3f}±{cv_res['test_precision'].std():.3f}"
          f" {cv_res['test_recall'].mean():.3f}±{cv_res['test_recall'].std():.3f}")

## Feature importance

In [ ]:
rf = RandomForestClassifier(500, random_state=42).fit(X_s, labels)
importances = rf.feature_importances_
pc_names = ["MW","LogP","TPSA","HBD","HBA","RotBonds","ArRings","MR","CSP3","ValElec"]

# PC features vs FP features
pc_total = importances[2048:].sum()
fp_total = importances[:2048].sum()
print(f"Fingerprint bits total importance:      {fp_total:.3f} ({fp_total*100:.1f}%)")
print(f"Physicochemical features total import.: {pc_total:.3f} ({pc_total*100:.1f}%)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(["Fingerprint (2048 bits)", "Physicochemical (10)"], [fp_total, pc_total],
        color=["#1565c0","#00897b"])
ax1.set_title("Feature group importance (RF)"); ax1.set_ylabel("Total importance")

pc_imp = importances[2048:]
ax2.barh(pc_names, pc_imp, color="#7b1fa2")
ax2.set_title("Individual physicochemical importances")
plt.tight_layout(); plt.savefig("feature_importance.png", dpi=150); plt.show()

## Key takeaways
- RF + Morgan ECFP4 is the standard baseline for molecular property prediction
- Always use stratified k-fold — class imbalance is common in bioactivity datasets
- Physicochemical features add interpretability; fingerprints add predictive power
- Compare AUC, F1, precision, and recall — AUC alone misses class imbalance problems